In [15]:
import numpy as np
import pandas as pd
import seaborn as sns
import pyrepseq as prs
from pathlib import Path
from collections import Counter
import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri, conversion, default_converter
from rpy2.robjects.packages import importr, isinstalled

In [16]:
utils = importr('utils') # R's base utilities

if not isinstalled('remotes'): # Check if the things work so that i can steal SPANTCR from github
    utils.install_packages('remotes', repos='https://cloud.r-project.org')

remotes = importr('remotes')
remotes.install_url("https://github.com/alexandermxu/SPANTCR/archive/refs/heads/main.zip")

R callback write-console: Downloading package from url: https://github.com/alexandermxu/SPANTCR/archive/refs/heads/main.zip
  


These packages have more recent versions available.
It is recommended to update all of them.
Which would you like to update?

1: All                          
2: CRAN packages only           
3: None                         
4: rlang (1.2.0 -> 1.3.0) [CRAN]

── R CMD build ─────────────────────────────────────────────────────────────────
* checking for file ‘/private/var/folders/mk/phnxp5857gsdgz1fz3lyrbjc0000gn/T/RtmpMSp9HR/remotesc3327a720d0d/SPANTCR-main/DESCRIPTION’ ... OK
* preparing ‘SPANTCR’:
* checking DESCRIPTION meta-information ... OK
* checking for LF line-endings in source and make files and shell scripts
* checking for empty or unneeded directories
* building ‘SPANTCR_0.0.0.9000.tar.gz’



* installing *source* package ‘SPANTCR’ ...
** this is package ‘SPANTCR’ version ‘0.0.0.9000’
** using staged installation
** R
** data
*** moving datasets to lazyload DB
** byte-compile and prepare package for lazy loading
** help
*** installing help indices
*** copying figures
** building package indices
** testing if installed package can be loaded from temporary location
** testing if installed package can be loaded from final location
** testing if installed package keeps a record of temporary installation path
* DONE (SPANTCR)


'SPANTCR'


In [17]:
with conversion.localconverter(default_converter + pandas2ri.converter):
    spantcr = importr('SPANTCR')
    base = importr('base')

exposed_objects = [name for name in dir(spantcr) if not name.startswith('-')]
print(exposed_objects)

['CDR3Breakdown', 'ComparisonFunction', 'EntropyScan', 'SPANTCR', 'ScoreFunctionExponential5', 'ScoreFunctionSquared', 'SearchIterator', 'WeightFunctionLinear', '___C__CDR3Breakdown', '___NAMESPACE___', '___S3MethodsTable___', '__doc__', '__loader__', '__name__', '__package__', '__rdata__', '__rname__', '__spec__', '__version__', '_env', '_exported_names', '_packageName', '_rpy2r', '_symbol_r2python', '_symbol_resolve', '_translation']


In [18]:
data_dir_kb = Path('../Data/20250910 Comparison 2/Kb')
data_dir_kd = Path('../Data/20250910 Comparison 2/Kd')
output_dir = Path('Comparison2_v2')
output_dir.mkdir(exist_ok=True)

In [19]:
kb_samples = {
    "1783 Naive LL": data_dir_kb / "20250910 1783 Naive LL TCR Repertoire.csv",
    "1783 Naive SLO": data_dir_kb / "20250910 1783 Naive SLO TCR Repertoire.csv",
    "B10BR NaiveA LL": data_dir_kb / "20250910 B10BR NaiveA LL TCR Repertoire.csv",
    "B10BR NaiveA SLO": data_dir_kb / "20250910 B10BR NaiveA SLO TCR Repertoire.csv",
    "B10BR NaiveB LL": data_dir_kb / "20250910 B10BR NaiveB LL TCR Repertoire.csv",
    "B10BR NaiveB SLO": data_dir_kb / "20250910 B10BR NaiveB SLO TCR Repertoire.csv",
    "B10BR PD1hiA LL": data_dir_kb / "20250910 B10BR PD1hiA LL TCR Repertoire.csv",
    "B10BR PD1hiB LL": data_dir_kb / "20250910 B10BR PD1hiB LL TCR Repertoire.csv",
    "B10BR PD1hiC LL": data_dir_kb / "20250910 B10BR PD1hiC LL TCR Repertoire.csv",
    "B10BR PD1hiD LL": data_dir_kb / "20250910 B10BR PD1hiD LL TCR Repertoire.csv",
    "B10BR PD1hiE LL": data_dir_kb / "20250910 B10BR PD1hiE LL TCR Repertoire.csv",
    "B10BR PD1negD LL": data_dir_kb / "20250910 B10BR PD1negD LL TCR Repertoire.csv",
    "B10BR Pre-immune": data_dir_kb / "20250910 B10BR pre-immmune TCR Repertoire.csv",
}
kd_samples = {
    "B6Kd Naive LL": data_dir_kd / "20250910 B6Kd Naive LL TCR Repertoire.csv",
    "B6Kd Naive SLO": data_dir_kd / "20250910 B6Kd Naive SLO TCR Repertoire.csv",
    "BL6 NaiveA LL": data_dir_kd / "20250910 BL6 NaiveA LL TCR Repertoire.csv",
    "BL6 NaiveA SLO": data_dir_kd / "20250910 BL6 NaiveA SLO TCR Repertoire.csv",
    "BL6 NaiveB LL": data_dir_kd / "20250910 BL6 NaiveB LL TCR Repertoire.csv",
    "BL6 NaiveB SLO": data_dir_kd / "20250910 BL6 NaiveB SLO TCR Repertoire.csv",
    "BL6 PD1hiA LL": data_dir_kd / "20251120 BL6 PD1hiA LL TCR Repertoire.csv",
    "BL6 PD1hiB LL": data_dir_kd / "20251120 BL6 PD1hiB LL TCR Repertoire.csv",
    "BL6 PD1hiC LL": data_dir_kd / "20251120 BL6 PD1hiC LL TCR Repertoire.csv",
    "BL6 PD1negA LL": data_dir_kd / "20250910 BL6 PD1negA LL TCR Repertoire.csv",
    "BL6 PD1negB LL": data_dir_kd / "20250910 BL6 PD1negB LL TCR Repertoire.csv",
    "BL6 PD1negC LL": data_dir_kd / "20250910 BL6 PD1negC LL TCR Repertoire.csv",
    "BL6 PD1negD LL": data_dir_kd / "20251120 BL6 PD1negD LL TCR Repertoire.csv",
    "C57BL6 Pre-immune SLO": data_dir_kd / "20250910 C57BL6 Pre-Immune SLO TCR Repertoire.csv",
}


In [20]:
kb_reactive = pd.concat([pd.read_csv(kb_samples["B10BR PD1hiA LL"]), pd.read_csv(kb_samples["B10BR PD1hiB LL"]),
                         pd.read_csv(kb_samples["B10BR PD1hiC LL"]), pd.read_csv(kb_samples["B10BR PD1hiD LL"]),
                         pd.read_csv(kb_samples["B10BR PD1hiE LL"])], ignore_index=True)

kb_selfTolerant = pd.concat([pd.read_csv(kb_samples["1783 Naive LL"]), pd.read_csv(kb_samples["1783 Naive SLO"])], ignore_index=True)

kb_bystander = pd.read_csv(kb_samples["B10BR PD1negD LL"])

kb_preImmune = pd.concat([pd.read_csv(kb_samples["B10BR NaiveA LL"]), pd.read_csv(kb_samples["B10BR NaiveA SLO"]),
                         pd.read_csv(kb_samples["B10BR NaiveB LL"]), pd.read_csv(kb_samples["B10BR NaiveB SLO"]),
                         pd.read_csv(kb_samples["B10BR Pre-immune"])], ignore_index=True)

In [21]:
kd_reactive = pd.concat([pd.read_csv(kd_samples["BL6 PD1hiA LL"]), pd.read_csv(kd_samples["BL6 PD1hiB LL"]),
                         pd.read_csv(kd_samples["BL6 PD1hiC LL"])], ignore_index=True)

kd_selfTolerant = pd.concat([pd.read_csv(kd_samples["B6Kd Naive LL"]), pd.read_csv(kd_samples["B6Kd Naive SLO"])], ignore_index=True)

kd_bystander = pd.concat([pd.read_csv(kd_samples["BL6 PD1negA LL"]), pd.read_csv(kd_samples["BL6 PD1negB LL"]),
                          pd.read_csv(kd_samples["BL6 PD1negC LL"]), pd.read_csv(kd_samples["BL6 PD1negD LL"])], ignore_index=True)

kd_preImmune = pd.concat([pd.read_csv(kd_samples["BL6 NaiveA LL"]), pd.read_csv(kd_samples["BL6 NaiveA SLO"]),
                         pd.read_csv(kd_samples["BL6 NaiveB LL"]), pd.read_csv(kd_samples["BL6 NaiveB SLO"]),
                         pd.read_csv(kd_samples["C57BL6 Pre-immune SLO"])], ignore_index=True)

In [22]:
kb_reactive_alpha = pd.DataFrame({'CDR3': kb_reactive['CDR3a_aa'], 'gene': 'TRA', 'Vgene': kb_reactive['TRAV'], 'Jgene': kb_reactive['TRAJ'], 'score': 1.0, 'id': kb_reactive['cell_id']})
kb_reactive_beta = pd.DataFrame({'CDR3': kb_reactive['CDR3b_aa'], 'gene': 'TRB', 'Vgene': kb_reactive['TRBV'], 'Jgene': kb_reactive['TRBJ'], 'score': 1.0, 'id': kb_reactive['cell_id']})
kb_r_paired = pd.concat([kb_reactive_alpha, kb_reactive_beta], ignore_index=True)

kb_selfTolerant_alpha = pd.DataFrame({'CDR3': kb_selfTolerant['CDR3a_aa'], 'gene': 'TRA', 'Vgene': kb_selfTolerant['TRAV'], 'Jgene': kb_selfTolerant['TRAJ'], 'score': 1.0, 'id': kb_selfTolerant['cell_id']})
kb_selfTolerant_beta = pd.DataFrame({'CDR3': kb_selfTolerant['CDR3b_aa'], 'gene': 'TRB', 'Vgene': kb_selfTolerant['TRBV'], 'Jgene': kb_selfTolerant['TRBJ'], 'score': 1.0, 'id': kb_selfTolerant['cell_id']})
kb_sT_paired = pd.concat([kb_selfTolerant_alpha, kb_selfTolerant_beta], ignore_index=True)

kb_preImmune_alpha = pd.DataFrame({'CDR3': kb_preImmune['CDR3a_aa'], 'gene': 'TRA', 'Vgene': kb_preImmune['TRAV'], 'Jgene': kb_preImmune['TRAJ'], 'score': 1.0, 'id': kb_preImmune['cell_id']})
kb_preImmune_beta = pd.DataFrame({'CDR3': kb_preImmune['CDR3b_aa'], 'gene': 'TRB', 'Vgene': kb_preImmune['TRBV'], 'Jgene': kb_preImmune['TRBJ'], 'score': 1.0, 'id': kb_preImmune['cell_id']})
kb_pI_paired = pd.concat([kb_preImmune_alpha, kb_preImmune_beta], ignore_index=True)

In [23]:
kd_reactive_alpha = pd.DataFrame({'CDR3': kd_reactive['CDR3a_aa'], 'gene': 'TRA', 'Vgene': kd_reactive['TRAV'], 'Jgene': kd_reactive['TRAJ'], 'score': 1.0, 'id': kd_reactive['cell_id']})
kd_reactive_beta = pd.DataFrame({'CDR3': kd_reactive['CDR3b_aa'], 'gene': 'TRB', 'Vgene': kd_reactive['TRBV'], 'Jgene': kd_reactive['TRBJ'], 'score': 1.0, 'id': kd_reactive['cell_id']})
kd_r_paired = pd.concat([kd_reactive_alpha, kd_reactive_beta], ignore_index=True)

kd_selfTolerant_alpha = pd.DataFrame({'CDR3': kd_selfTolerant['CDR3a_aa'], 'gene': 'TRA', 'Vgene': kd_selfTolerant['TRAV'], 'Jgene': kd_selfTolerant['TRAJ'], 'score': 1.0, 'id': kd_selfTolerant['cell_id']})
kd_selfTolerant_beta = pd.DataFrame({'CDR3': kd_selfTolerant['CDR3b_aa'], 'gene': 'TRB', 'Vgene': kd_selfTolerant['TRBV'], 'Jgene': kd_selfTolerant['TRBJ'], 'score': 1.0, 'id': kd_selfTolerant['cell_id']})
kd_sT_paired = pd.concat([kd_selfTolerant_alpha, kd_selfTolerant_beta], ignore_index=True)

kd_preImmune_alpha = pd.DataFrame({'CDR3': kd_preImmune['CDR3a_aa'], 'gene': 'TRA', 'Vgene': kd_preImmune['TRAV'], 'Jgene': kd_preImmune['TRAJ'], 'score': 1.0, 'id': kd_preImmune['cell_id']})
kd_preImmune_beta = pd.DataFrame({'CDR3': kd_preImmune['CDR3b_aa'], 'gene': 'TRB', 'Vgene': kd_preImmune['TRBV'], 'Jgene': kd_preImmune['TRBJ'], 'score': 1.0, 'id': kd_preImmune['cell_id']})
kd_pI_paired = pd.concat([kd_preImmune_alpha, kd_preImmune_beta], ignore_index=True)

In [24]:
with conversion.localconverter(default_converter + pandas2ri.converter):

    r_kb_r = conversion.py2rpy(kb_r_paired)
    r_kb_sT = conversion.py2rpy(kb_sT_paired)
    r_kb_pI = conversion.py2rpy(kb_pI_paired)

    r_kd_r = conversion.py2rpy(kd_r_paired)
    r_kd_sT = conversion.py2rpy(kd_sT_paired)
    r_kd_pI = conversion.py2rpy(kd_pI_paired)

robjects.r.assign('kb_r_raw', r_kb_r)
robjects.r.assign('kb_sT_raw', r_kb_sT)
robjects.r.assign('kb_pI_raw', r_kb_pI)

robjects.r.assign('kd_r_raw', r_kd_r)
robjects.r.assign('kd_sT_raw', r_kd_sT)
robjects.r.assign('kd_pI_raw', r_kd_pI)

robjects.r('''
    library(data.table)
    library(SPANTCR)

    kb_r_dt <- as.data.table(kb_r_raw)
    kb_r <- SPANTCR(kb_r_dt, 'kb_r_span', 'Paired', 'Hydrophobicity', 100, 3, 0.03, ScoreFunctionExponential5, WeightFunctionLinear)
    ''')

In [25]:
robjects.r('''
library(data.table)
library(SPANTCR)

kb_sT_dt <- as.data.table(kb_sT_raw)
kb_sT <- SPANTCR(kb_sT_dt, 'kb_sT_span', 'Paired', 'Hydrophobicity', 100, 3, 0.03, ScoreFunctionExponential5, WeightFunctionLinear)
''')

In [26]:
robjects.r('''
library(data.table)
library(SPANTCR)

kd_r_dt <- as.data.table(kd_r_raw)
kd_r <- SPANTCR(kd_r_dt, 'kd_r_span', 'Paired', 'Hydrophobicity', 100, 3, 0.03, ScoreFunctionExponential5, WeightFunctionLinear)
''')

In [27]:
robjects.r('''
library(data.table)
library(SPANTCR)

kd_sT_dt <- as.data.table(kd_sT_raw)
kd_sT <- SPANTCR(kd_sT_dt, 'kd_sT_span', 'Paired', 'Hydrophobicity', 100, 3, 0.03, ScoreFunctionExponential5, WeightFunctionLinear)
''')

In [ ]:
robjects.r('''
library(data.table)
library(SPANTCR)

kb_pI_dt <- as.data.table(kb_pI_raw)
kb_pI <- SPANTCR(kb_pI_dt, 'kb_pI_span', 'Paired', 'Hydrophobicity', 100, 3, 0.03, ScoreFunctionExponential5, WeightFunctionLinear)
''')

In [ ]:
robjects.r('''
library(data.table)
library(SPANTCR)

kd_pI_dt <- as.data.table(kd_pI_raw)
kd_pI <- SPANTCR(kd_pI_dt, 'kd_pI_span', 'Paired', 'Hydrophobicity', 100, 3, 0.03, ScoreFunctionExponential5, WeightFunctionLinear)
''')

In [ ]:
robjects.r('''
CMVr_sT_Distance <- ComparisonFunction(kb_r, kb_sT, 100, FineBinTicksPaired, BLOSUMCappedDiff, F)
#> 0.10/ 0.20/ 0.30/ 0.40/ 0.50/ 0.60/ 0.70/ 0.80/ 0.90/ 1.00/ 1.10/ 1.20/ 1.30/ 1.40/ 1.50/ 1.60/ 1.70/ 1.80/ 1.90/ 2.00

CMVr_sT_Distance[, SecondChain := sapply(Top5Chain, function(x) x[2])]
CMVr_sT_Distance[, SecondChainScore := sapply(Top5CompScore, function(x) x[2])]
CMVr_sT_Distance[, SecondChainLev := diag(adist(SecondChain, BaseCDR3))]

d <- ggplot(CMVr_sT_Distance)+
  geom_point(aes(x=SecondChainLev, y=SecondChainScore))+
  coord_fixed()

ggsave("kb_r_sT_distance.png", plot = d, width = 8, height = 6, dpi = 300)
''')

0.005/ 0.015/ 0.025/ 0.035/ 0.045/ 0.055/ 0.065/ 0.075/ 0.085/ 

R callback write-console: Error: vector memory limit of 16.0 Gb reached, see mem.maxVSize()
  


RRuntimeError: Error: vector memory limit of 16.0 Gb reached, see mem.maxVSize()


In [ ]:
robjects.r('''
library(ggplot2)

a <- ggplot(kb_r@Output[order(Window)])+
  geom_bar(aes(x=Tick, y=WeightProbability, fill=SignificantColor), stat="identity", width=1/100)+
  theme(legend.position="none")+
  scale_fill_gradientn(colors=c(alpha("#4D4D4D",0.01),"red","white","blue"), values=c(0,0.01,0.50,1))+
  coord_fixed()

ggsave("kb_r_heatmap.png", plot = a, width = 8, height = 6, dpi = 300)
''')

In [ ]:
robjects.r('''
library(ggplot2)

b <- ggplot(kb_sT@Output[order(Window)])+
  geom_bar(aes(x=Tick, y=WeightProbability, fill=SignificantColor), stat="identity", width=1/100)+
  theme(legend.position="none")+
  scale_fill_gradientn(colors=c(alpha("#4D4D4D",0.01),"red","white","blue"), values=c(0,0.01,0.50,1))+
  coord_fixed()

ggsave("kb_sT_heatmap.png", plot = b, width = 8, height = 6, dpi = 300)
''')

In [ ]:
robjects.r('''
SearchBoxesPaired20 <- mapply(c, seq(0,1.9,length.out=20), seq(0.1,2,length.out=20), SIMPLIFY=F)
VDJDBCMV_Entropy1 <- EntropyScan(kb_r, SearchBoxesPaired20, 0.03, FineBinTicksPaired)

VDJDBCMV_Entropy1[[1]]$Source <- factor(VDJDBCMV_Entropy1[[1]]$Source, levels=c("Base", sort(unique(VDJDBCMV_Entropy1[[1]]$Source)[-1])))

e <- ggplot(VDJDBCMV_Entropy1[[1]][Count>20])+
geom_tile(aes(x=Range, y=Source, fill=DeltaAverage))+
scale_x_discrete(breaks=c("0-0.1","1-1.1","1.9-2"), labels=c(0,1,2))+
scale_fill_gradient2(low="blue", mid="white", high="red", midpoint = 1)+
theme_dark()

ggsave("kb_r_entropyscan.png", plot = e, width = 8, height = 6, dpi = 300)
''')

In [ ]:
robjects.r('''

SearchBoxesPaired20 <- mapply(c, seq(0,1.9,length.out=20), seq(0.1,2,length.out=20), SIMPLIFY=F)
VDJDBCMV_Entropy2 <- EntropyScan(kb_sT, SearchBoxesPaired20, 0.03, FineBinTicksPaired)

VDJDBCMV_Entropy2[[1]]$Source <- factor(VDJDBCMV_Entropy2[[1]]$Source, levels=c("Base", sort(unique(VDJDBCMV_Entropy2[[1]]$Source)[-1])))

f <- ggplot(VDJDBCMV_Entropy2[[1]][Count>20])+
geom_tile(aes(x=Range, y=Source, fill=DeltaAverage))+
scale_x_discrete(breaks=c("0-0.1","1-1.1","1.9-2"), labels=c(0,1,2))+
scale_fill_gradient2(low="blue", mid="white", high="red", midpoint = 1)+
theme_dark()

ggsave("kb_sT_entropyscan.png", plot = f, width = 8, height = 6, dpi = 300)
''')

In [ ]:
robjects.r('''
g <- ggplot(VDJDBCMV_Entropy1[[2]][N>10 & Range=="0.5-0.6"])+
  geom_line(aes(x=Location, y=Entropy, color=Source))+
  theme_bw()

ggsave("kb_r_geom.png", plot = g, w idth = 8, height = 6, dpi = 300)
''')

In [ ]:
robjects.r('''
h <- ggplot(VDJDBCMV_Entropy2[[2]][N>10 & Range=="0.5-0.6"])+
  geom_line(aes(x=Location, y=Entropy, color=Source))+
  theme_bw()

ggsave("kb_sT_geom.png", plot = h, width = 8, height = 6, dpi = 300)
''')

In [ ]:
kb_reactive_spantcr = robjects.r('kb_r')
kb_selfTolerant_spantcr = robjects.r('kb_sT') 
kb_preImmune_spantcr = robjects.r('kb_pI')

kd_reactive_spantcr = robjects.r('kd_r')
kd_selfTolerant_spantcr = robjects.r('kd_sT')
kd_preImmune_spantcr = robjects.r('kd_pI')